In [104]:
import re

# =========================
# 1. CUSTOM STOPWORDS (ISI SENDIRI)
# =========================
custom_stopwords = set([
    # isi sendiri nanti
    "yang",
    "dan",
    "di",
    "ke",
    "dari",
    "ini",
    "itu",
    "gak",
    "ga",
    "saya",
    "kamu",
    "dia",
    "kami",
    "mereka",
    "ya",
    "ad",
    "jual",
    "dan",
    "dari",
    "yg",
    "byk",
    "query",
    "nya",
    "tidak",
    "dokter",
    "doi",
    "sm",
    "mas",
    "mbak",
    "gw",
    "bisa",
    "klo",
    "udah",
    "ada",
    "aja",
    "gue",
    "meski",
    "tidak",
    "aja",
    "dah",
    "pilih"
    "gk",
    "shg",
    "ntar"
    "jt",
    "gk",
    "lu",
    "atau"
    "dr"
])

# =========================
# 2. CLEAN FUNCTION
# =========================
def clean_text(text):
    text = str(text).lower()

    # hapus url
    text = re.sub(r"http\S+|www\S+", "", text)

    # hapus mention & hashtag
    text = re.sub(r"@\w+|#\w+", "", text)

    # hapus angka & simbol
    text = re.sub(r"[^a-z\s]", " ", text)

    # rapihin spasi
    text = re.sub(r"\s+", " ", text).strip()

    # hapus stopwords custom
    words = text.split()
    words = [w for w in words if w not in custom_stopwords]

    return " ".join(words)

In [105]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN

# =========================
# 1. Load data
# =========================
df = pd.read_csv("hasil_labeling_ev - NEGATIF 800-2.csv")
df["clean_textdisplay"] = df["clean_textdisplay"].astype(str).apply(clean_text)

docs = df["clean_textdisplay"].fillna("").astype(str).tolist()

In [106]:
print(df.columns)

Index(['date', 'authordisplayname', 'textdisplay', 'likecount',
       'clean_textdisplay', 'label', 'confidence', 'status', 'Unnamed: 8',
       'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19'],
      dtype='object')


In [107]:
# =========================
# 2. SOTA EMBEDDING MODEL
# =========================
model = SentenceTransformer("intfloat/multilingual-e5-large")

# E5 BUTUH PREFIX (IMPORTANT!)
embeddings = model.encode(
    ["query: " + d for d in docs],   # hanya untuk embedding
    show_progress_bar=True
)


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

In [108]:
# =========================
# 3. REDUCE DIMENSION (UMAP)
# =========================
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

In [109]:
hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [110]:
# =========================
# 5. BERTopic MODEL
# =========================
topic_model = BERTopic(
    top_n_words=10,
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)

2026-06-20 13:09:05,696 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-20 13:09:06,624 - BERTopic - Dimensionality - Completed ✓
2026-06-20 13:09:06,624 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-20 13:09:06,635 - BERTopic - Cluster - Completed ✓
2026-06-20 13:09:06,637 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-20 13:09:06,651 - BERTopic - Representation - Completed ✓


In [111]:
# =========================
# 6. TRAIN MODEL
# =========================
topics, probs = topic_model.fit_transform(docs)

2026-06-20 13:09:06,671 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

2026-06-20 13:09:15,725 - BERTopic - Embedding - Completed ✓
2026-06-20 13:09:15,725 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-20 13:09:16,515 - BERTopic - Dimensionality - Completed ✓
2026-06-20 13:09:16,515 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-20 13:09:16,523 - BERTopic - Cluster - Completed ✓
2026-06-20 13:09:16,524 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-20 13:09:16,538 - BERTopic - Representation - Completed ✓


In [123]:
df = topic_model.get_document_info(docs)

df.to_csv("topic_training_info.csv", index=False)

In [112]:
topic_model.visualize_topics()

In [121]:
topic_model.get_representative_docs(10)
topic_model.get_representative_docs(14)

['perkara dompet kalo rugi artinya dompetmu belum siapbritu sama macem lebih mending beli brio pada merci e jangan beli mercy karena rugi biayanya pasti banyak keluar service mahal bensin seirit briobrbritu menandakan dompetmu belum berotot bang mungkin masih level ekonomi rata orang indonesia yaitu motor beat bekas real rata ekonomi indonesia bergdp hanya alias gaji dibawah jutabrjangan makan hotel mahal toh sama juga makanannya bakal jadi tai juga mending makan warteg murah meriah kenyangbranda dompet berlum berotot tapi lagu cepat bener',
 'gundala kereta cepat whoosh memiliki utang besar sumber utamanya china development bank cdb untuk membiayai pembangunan dengan total biaya membengkak menjadi sekitar rp triliun beban ditanggung oleh pt kereta cepat indonesia china kcic konsorsium bumn indonesia psbi pihak tiongkok sedang dinegosiasi ulang tenornya agar lebih ringan brbrbiar bego',
 'gundalani orang belajar sih terus modal bumn mana kalau salah satunya suntikan pemerintah bahkan p

In [113]:
topic_model.visualize_barchart(top_n_topics=50, n_words=10)

In [114]:
# =========================
# 8. TOPIC SUMMARY
# =========================
print(topic_model.get_topic_info())


    Topic  Count                                  Name  \
0      -1    224     -1_mobil_listrik_vehicle_electric   
1       0    116               0_harga_brio_atto_honda   
2       1    103           1_baterai_harga_mobil_tahun   
3       2     49        2_electric_vehicle_air_listrik   
4       3     46           3_bensin_minyak_bakar_mobil   
5       4     37             4_china_jepang_cina_mobil   
6       5     36    5_listrik_tambang_merusak_batubara   
7       6     30               6_kota_mobil_macet_buat   
8       7     29             7_mobil_dijual_brio_masih   
9       8     26        8_electric_vehicle_tahun_harga   
10      9     23     9_hidrogen_hydrogen_energi_jepang   
11     10     22  10_pemerintah_bantuan_pajak_regulasi   
12     11     18                  11_ice_aki_mobil_ban   
13     12     16             12_charger_rb_portable_jt   
14     13     13   13_listrik_narasumber_mobil_bengkel   
15     14     12          14_indonesia_bumn_rata_pajak   

             

In [115]:
df_pred = pd.read_csv("hasil_sentimen.predictions - negative.csv")

new_docs = df_pred["clean_textdisplay"].astype(str).tolist()

In [116]:
pred_topics, pred_probs = topic_model.transform(new_docs)

Batches:   0%|          | 0/21 [00:00<?, ?it/s]

2026-06-20 13:09:17,318 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-06-20 13:09:17,696 - BERTopic - Dimensionality - Completed ✓
2026-06-20 13:09:17,697 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-06-20 13:09:17,704 - BERTopic - Cluster - Completed ✓


In [117]:
df_pred["pred_topic"] = pred_topics

In [118]:
topic_info = topic_model.get_topic_info()
topic_names = dict(zip(topic_info["Topic"], topic_info["Name"]))
df_pred["topic_name"] = df_pred["pred_topic"].map(topic_names)


In [119]:
print(df_pred.head())

                        date     authordisplayname  \
0  2024-10-26 01:55:33+00:00               @kedepp   
1  2024-10-26 01:02:47+00:00  @adolfbabehdolof8546   
2  2024-10-25 23:48:20+00:00              @guz3108   
3  2024-10-26 01:27:38+00:00      @aanganggara1699   
4  2024-10-25 23:12:27+00:00    @wawansetiawan3474   

                                         textdisplay  likecount  \
0  Jikanper hari isi bensin 100ribu, setahun 360 ...        0.0   
1  Konten negatif, selalu menjelekkan EV, karena ...        6.0   
2  Harga jual itu soal supply demand. Sekarang ke...        1.0   
3  Anjiir murah amat klo 500rb sebulan, bensin mo...        0.0   
4  Gak fair sih klo penilaian hanya di harga jual...        2.0   

                                   clean_textdisplay    label  confidence  \
0  jikanper hari isi bensin ribu setahun hari ber...  Positif        0.86   
1  konten negatif selalu menjelekkan electric veh...  Positif        0.91   
2  harga jual itu soal supply demand seka

In [120]:
df_pred.to_csv("hasil_prediksi_topik.csv", index=False)